# Project Cycle 3: Two-sample inference
**Team Name:** [第26組]

**Team Members (組員姓名):** [111A50012 楊書雅], [113370216	彭佳淳]  


## Step 1.1: Introduction & Research Questions
我們小組選擇了**問題 3**進行深入研究。

* **選定研究問題 (Research Question):** 在男學生與女學生之間，感到悲傷或絕望的學生比例是否有顯著差異？
* **組別變數 (Group Variable):** `WhatIsYourSex` (性別，分為 Male 與 Female 兩組)
* **反應變數 (Response Variable):** `SadOrHopeless` (是否感到悲傷或絕望)
* **統計方法 (Method Used):** 雙比例 z 檢定 (Two-proportion z-test) 及 兩比例差異的信賴區間 (Confidence interval for difference in proportions)。
* **最終結論摘要 (Short Final Conclusion):** *[註：此處待你們跑完程式碼後根據 p-value 填寫。例如：本研究在 alpha = 0.05 的顯著水準下，拒絕虛無假設。結果顯示女性學生感到悲傷或絕望的比例顯著高於男性學生。]*

---

## 📌 Research Question Overview

Our group has selected **Question 3** for this research project.

* **Selected Research Question:** Is the proportion of students who felt sad or hopeless different between male and female students?
* **Group Variable:** `WhatIsYourSex` (Gender, divided into Male and Female groups)
* **Response Variable:** `SadOrHopeless` (Feeling sad or hopeless)
* **Statistical Method:** Two-proportion z-test and Confidence interval for the difference in proportions.
* **Short Final Conclusion:** *[Note: To be filled in based on your p-value after running the code. For example: At the alpha = 0.05 significance level, we reject the null hypothesis. The results indicate that the proportion of female students who felt sad or hopeless is significantly higher than that of male students.]*

## 1.2 Setup & Data Loading

### 資料載入與套件準備 / Data Loading and Package Setup

在此步驟中，我們將載入資料分析所需的 Python 核心套件，包括用於資料處理的 `pandas`、用於統計檢定的 `statsmodels`，以及用於繪圖的 `matplotlib` 與 `seaborn`。接著，我們將從相對路徑 `data/raw/` 讀取原始的 YRBS 2007 年調查數據集。


In this step, we import the core Python libraries required for data analysis, including `pandas` for data manipulation, `statsmodels` for statistical testing, and `matplotlib` along with `seaborn` for data visualization. We then load the original YRBS 2007 dataset from the relative path `data/raw/`.

In [11]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest

# 使用 '..' 代表往上一層，從 notebooks/ 離開，回到 project-cycle-3/ 主目錄
raw_data_path = os.path.join('..', 'data', 'raw', 'YRBS_2007.csv')

# 自動檢查修正後的路徑
if not os.path.exists(raw_data_path):
    print(f"❌ 錯誤：即使使用了相對路徑，在 '{raw_data_path}' 仍找不到檔案！")
    print(f"🔍 目前工作目錄：{os.getcwd()}")
else:
    df = pd.read_csv(raw_data_path)
    print(f"✅ 原始數據集載入成功！")
    print(f"📊 資料集形狀：總計 {df.shape[0]} 行, {df.shape[1]} 列。")

✅ 原始數據集載入成功！
📊 資料集形狀：總計 14041 行, 103 列。


原始數據集已順利載入。系統顯示資料包含完整的受訪學生觀測值。我們已將資料妥善保存在 data/raw/ 中，絕不對其進行直接修改，以確保資料的原始性與可重現性。下一階段我們將進行專門的變數篩選與清理。

The raw dataset has been successfully loaded. The statistics indicate a comprehensive collection of survey responses. The raw file remains untouched in data/raw/ to ensure baseline data integrity and reproducibility. Next, we will proceed to specific variable selection and data cleaning.

## 1.3 變數篩選與資料清理 / Variable Selection and Data Cleaning

本研究聚焦於性別與悲傷絕望感之間的關係（對應研究問題 3）。因此，我們從大數據集中提取與本專案核心相關的兩個關鍵欄位：

代表性別的組別變數 WhatIsYourSex 與代表悲傷絕望感的反應變數 SadOrHopeless。為了確保後續雙樣本比例檢定的準確性與有效性，我們必須排除並過濾掉在這兩個變數中含有遺漏值（NaN）的無效觀測值。

This study focuses on the relationship between gender and feelings of sadness or hopelessness (corresponding to Research Question 3). Therefore, we extract the two critical columns relevant to our project: 

The group variable WhatIsYourSex and the response variable SadOrHopeless. To ensure the accuracy and validity of our subsequent two-sample proportion test, we must filter out and remove invalid observations containing missing values (NaN) in either variable.

In [12]:
# 篩選特定目標變數並複製資料集
analysis_df = df[['WhatIsYourSex', 'SadOrHopeless']].copy()

# 檢查清理前的遺漏值數量
print("清理前各欄位遺漏值 (Missing values) 個數：")
print(analysis_df.isnull().sum())

# 移除任一欄位含有缺失值的資料列 (dropna)
analysis_df.dropna(subset=['WhatIsYourSex', 'SadOrHopeless'], inplace=True)

print(f"\n✅ 清理缺失值後，用於分析的有效樣本數為：{analysis_df.shape[0]} 筆。")

清理前各欄位遺漏值 (Missing values) 個數：
WhatIsYourSex     13
SadOrHopeless    196
dtype: int64

✅ 清理缺失值後，用於分析的有效樣本數為：13833 筆。


我們已成功篩選出目標變數並過濾掉所有缺失值，留下的是具有完整回答的有效樣本。這能有效避免未定義的缺失值在後續統計分析與編碼時造成程式錯誤或計算偏差。


Explanation: We have successfully isolated our target variables and dropped all records with missing values, leaving only valid responses with complete information. This step prevents undefined missing data from causing program errors or biasing our subsequent statistical analyses and recoding.

## 1.4 Data Preparation and Recoding / 數據準備與變數重編碼

在執行統計推論之前，將原始類別代碼轉換為明確的二元指標至關重要，能有效避免將代碼誤視為連續數字進行計算：
1. **組別變數**：將性別的原始數字代碼直接映射為明確的字串標籤（`Male` 與 `Female`），以便進行清晰的分組。
2. **反應變數**：將「成功（是）」定義為曾感受到悲傷或絕望，由原始代碼 1 重編碼為 `1`；「失敗（否）」則由原始代碼 2 重編碼為 `0`。
3. **儲存**：處理完成的乾淨數據集將透過相對路徑導出至 `data/processed/`，以確保研究的重現性。

本研究聚焦於性別與悲傷絕望感之間的關係（對應研究問題 3）。因此，我們從大數據集中提取與本專案核心相關的兩個關鍵欄位：

代表性別的組別變數 WhatIsYourSex 與代表悲傷絕望感的反應變數 SadOrHopeless。為了確保後續雙樣本比例檢定的準確性與有效性，我們必須排除並過濾掉在這兩個變數中含有遺漏值（NaN）的無效觀測值。

Before conducting statistical inference, transforming raw categorical data into explicit binary indicators is necessary to prevent numerical calculation errors on codes:
1. **Group Variable**: Raw numerical codes for sex are mapped directly to distinct string labels (`Male` and `Female`) for explicit grouping.
2. **Response Variable**: Success (Yes) is defined as having experienced feelings of sadness or hopelessness, recoded from code 1 to `1`. Failure (No) is recoded from code 2 to `0`.
3. **Storage**: The finalized clean dataset is exported to `data/processed/` using relative paths to ensure reproducibility.

This study focuses on the relationship between gender and feelings of sadness or hopelessness (corresponding to Research Question 3). Therefore, we extract the two critical columns relevant to our project: 

The group variable WhatIsYourSex and the response variable SadOrHopeless. To ensure the accuracy and validity of our subsequent two-sample proportion test, we must filter out and remove invalid observations containing missing values (NaN) in either variable.

In [13]:
import pandas as pd
import numpy as np
import os

print("--- 1. 資料載入階段 ---")
# 使用 '..' 離開 notebooks/ 檔案夾，確保能正確對齊主目錄的資料分層
raw_data_path = os.path.join('..', 'data', 'raw', 'YRBS_2007.csv')

if not os.path.exists(raw_data_path):
    print(f"❌ 錯誤：在相對路徑 '{raw_data_path}' 找不到原始檔案！")
    print(f"🔍 目前程式運行的工作目錄為：{os.getcwd()}")
else:
    # 載入原始資料
    df = pd.read_csv(raw_data_path)
    print(f"✅ 原始數據集載入成功！基本形狀：{df.shape[0]} 行, {df.shape[1]} 列。")

    print("\n--- 2. 資料清理階段 ---")
    # 篩選特定目標變數並複製資料集
    analysis_df = df[['WhatIsYourSex', 'SadOrHopeless']].copy()
    print("清理前各欄位遺漏值 (Missing values) 個數：")
    print(analysis_df.isnull().sum())

    # 移除任一欄位含有缺失值的資料列
    analysis_df.dropna(subset=['WhatIsYourSex', 'SadOrHopeless'], inplace=True)
    print(f"清理缺失值後，用於分析的有效樣本數為：{analysis_df.shape[0]} 筆。")

    print("\n--- 3. 變數重編碼階段 ---")
    # 重編碼組別變數 (1=Female, 2=Male)
    analysis_df['Gender'] = analysis_df['WhatIsYourSex'].map({1: 'Female', 2: 'Male'})
    # 重編碼反應變數 (Success/code 1 -> 1, Failure/code 2 -> 0)
    analysis_df['Sad_Recoded'] = analysis_df['SadOrHopeless'].map({1: 1, 2: 0})

    # 檢查確認重編碼分佈
    print("重編碼後的反應變數（0=無悲傷, 1=有悲傷）分佈：")
    print(analysis_df['Sad_Recoded'].value_counts(dropna=False))

    print("\n--- 4. 檔案分層儲存階段 ---")
    # 定義儲存路徑（回退一層到主目錄，再進入 data/processed）
    processed_dir = os.path.join('..', 'data', 'processed')
    processed_file_path = os.path.join(processed_dir, 'cleaned_sadness_data.csv')

    # 確保資料夾存在並儲存檔案
    os.makedirs(processed_dir, exist_ok=True)
    analysis_df.to_csv(processed_file_path, index=False)
    print(f"✅ 處理後的乾淨數據已成功儲存至: {processed_file_path}")

--- 1. 資料載入階段 ---
✅ 原始數據集載入成功！基本形狀：14041 行, 103 列。

--- 2. 資料清理階段 ---
清理前各欄位遺漏值 (Missing values) 個數：
WhatIsYourSex     13
SadOrHopeless    196
dtype: int64
清理缺失值後，用於分析的有效樣本數為：13833 筆。

--- 3. 變數重編碼階段 ---
重編碼後的反應變數（0=無悲傷, 1=有悲傷）分佈：
Sad_Recoded
0    9686
1    4147
Name: count, dtype: int64

--- 4. 檔案分層儲存階段 ---
✅ 處理後的乾淨數據已成功儲存至: ..\data\processed\cleaned_sadness_data.csv


### Verification of Data Preparation / 資料準備與重編碼結果驗證

**Explanation:** The complete ingestion, filtration, and transformation workflow has been successfully completed in a consolidated pipeline. The response variable is now formatted as a standard 0/1 binary indicator, and the group variable is clearly labeled. The resulting processed file is archived in `data/processed/`, securing a clean environment for subsequent statistical modeling without variable missingness or code reference issues.


**解釋：** 完整的資料載入、過濾與轉換流程已在整合管線中順利執行完畢。反應變數現在已轉化為標準的 0 與 1 二元指標，組別也已清晰標記。產出的處理後數據已妥善歸檔於 `data/processed/` 中，為下階段的統計建模奠定了乾淨、無缺失值干擾且無代碼混淆的分析環境。